# Task 3

In [ ]:
import HMM_inference
import matplotlib.pyplot as plt
import numpy as np
from HMM_models import RampModelHMM, StepModelHMM

In [ ]:
plt.rcParams['axes.titlesize'] = 18  # Set default title font size
plt.rcParams['figure.titlesize'] = 22
plt.rcParams['font.family'] = ['cmr10', 'Times New Roman', 'STIXGeneral']
plt.rcParams['xtick.labelsize'] = 18  # Font size of x-axis tick labels
plt.rcParams['ytick.labelsize'] = 18  # Font size of y-axis tick labels

## Task 3.1.1.

Visualising the approximate posterior on the grid, repeated for different true parameter values and for different numbers of trials for each model.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# Example varying parameter - here N changes
N_values = [1, 20, 100, 400]

for ax, N in zip(axs.flat, N_values):
    _ = HMM_inference.ramp_inference_scan(N=N,
                                        prior_type='uniform', 
                                        ax=ax,
                                        plot=True)
    ax.set_title(f'N={N}')


plt.suptitle('Ramp Model Posterior Scan for Different Sample Sizes \n (Flat Prior)', fontsize=22)
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)


N_values = [1, 20, 200, 400]
for ax, N in zip(axs.flat, N_values):
    _ = HMM_inference.step_inference_scan(true_m=200,
                                                    true_r=4,
                                                    ax=ax, 
                                                    prior_type='uniform',
                                                    N=N,
                                                    plot=True)
    ax.set_title(f'Sample Size={N}')

plt.suptitle('Step Model Posterior Scan for Different Sample Sizes \n (Flat Prior)', fontsize=22)
plt.show()

In [ ]:
fixed_N = 20
beta_true_values = [0.5, 1, 1.5]
sigma_true_values = [0.5, 1, 3]

fig, axs = plt.subplots(3, 3, figsize=(15, 12), constrained_layout=True)

for i, true_beta in enumerate(beta_true_values):
    for j, true_sigma in enumerate(sigma_true_values):
        ax = axs[i, j]
        _ = HMM_inference.ramp_inference_scan(
            true_beta=true_beta,
            true_sigma=true_sigma,
            prior_type='uniform',
            N=fixed_N,
            ax=ax,
            plot=True
        )
        ax.set_title(rf'Model: $\beta$={true_beta}, $\sigma$={true_sigma}')

plt.suptitle('Ramp Model Posterior for a range of Parameters', fontsize=22)
plt.show()


In [ ]:
fixed_N = 200
T = 500
m_true_values = [0.35*T, 0.5*T, 0.65*T]
r_true_values = [1, 3, 5]

fig, axs = plt.subplots(3, 3, figsize=(15, 12), constrained_layout=True)

for i, m in enumerate(m_true_values):
    for j, r in enumerate(r_true_values):
        ax = axs[i, j]
        _ = HMM_inference.step_inference_scan(
            true_m=m,
            true_r=r,
            N=fixed_N,
            prior_type='uniform',
            ax=ax
        )
        ax.set_title(rf'Model: m={m}, r={r}')

plt.suptitle('Step Model Posterior for a range of Parameters \n (Flat Prior)')
plt.show()


## Task 3.1.2.

Evaluation of posterior expectations of parameters, measuring error as difference between the true and predicted parameters and plotting how this error changes with number of trials and choices of true parameters.

### Posterior means and stds

In [ ]:
def posterior_stats(posterior, *axes):
    posterior = posterior / posterior.sum()
    means, stds = [], []
    ndim = posterior.ndim
    for d, grid in enumerate(axes):
        v = np.reshape(grid, [len(grid) if i == d else 1 for i in range(ndim)])
        m = (posterior * v).sum()
        s = np.sqrt((posterior * (v - m) ** 2).sum())
        means.append(m)
        stds.append(s)
    return means, stds


In [ ]:
import matplotlib.pyplot as plt

def collect_stats(scan_fn, N_vals):
    means, sds = [], []
    for N in N_vals:
        post, m_grid, r_grid, *_ = scan_fn(N=N, plot=False)
        m, s = posterior_stats(post, m_grid, r_grid)
        means.append(m)
        sds.append(s)
    return np.array(means).T, np.array(sds).T  # shape (n_params, n_N)

def plot_estimates(N_vals, means, sds, true_vals, titles, ylabel):
    fig, axes = plt.subplots(1, len(true_vals), figsize=(4*len(true_vals), 4), sharex=True)
    for ax, mu, sd, tru, title in zip(axes, means, sds, true_vals, titles):
        ax.errorbar(N_vals, mu, yerr=sd, fmt='o-', capsize=4)
        ax.axhline(tru, ls='--')
        ax.set_title(title)
        ax.set_xlabel('N')
        ax.set_ylabel(ylabel)
    fig.tight_layout()
    return fig, axes

# step model
N_vals = [1, 20, 100, 400]
step_means, step_sds = collect_stats(HMM_inference.step_inference_scan,
                                     N_vals)
plot_estimates(N_vals, step_means, step_sds,
               true_vals=(250, 3),
               titles=('m', 'r'),
               ylabel='posterior mean')

# ramp model
N_vals = [1, 20, 100, 400]
ramp_means, ramp_sds = collect_stats(HMM_inference.ramp_inference_scan,
                                     N_vals)
plot_estimates(N_vals, ramp_means, ramp_sds,
               true_vals=(0.5, 1),
               titles=('σ', 'β'),
               ylabel='posterior mean')


## Task 3.2. - Bayes factor calculation

Use a truncated Gaussian prior centred on the midpoint of the investigated range and attempt to classify using Bayes Factor (Marginal Likelihood Ratio)

# instead of having 'true params' we need to pass in the data generated above into these

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# vary the size of the gaussian prior
f = [0.1, 0.5, 2, 3]

for ax, sd in zip(axs.flat, f):
    _ = HMM_inference.ramp_inference_scan(ax=ax,
                                    N=20, 
                                    prior_type='gaussian',
                                    prior_sd_fraction=sd)
    ax.set_title(f'Gaussian Width = {sd}')

plt.suptitle('Ramp Model Posterior, varying width of central Gaussian prior')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# vary the size of the gaussian prior
f = [0.1, 0.5, 2, 3]

for ax, sd in zip(axs.flat, f):
    _ = HMM_inference.step_inference_scan(true_m=250,
                                                    ax=ax, 
                                                    prior_type='gaussian',
                                                    prior_sd_fraction=sd)
    ax.set_title(f'Gaussian Width = {sd}')

plt.suptitle('Step Model Posterior, varying width of central Gaussian prior')
plt.show()

Now we use the outputs of the functions to compare the Marginal Likelihoods and compare the Bayes' Factors

In [ ]:
# simulate some spike train ensembles from each model
K=100
s0=int(round(0.2*(K-1)))
fixed_rh = 50
T = 500
dt = 1/T
K=100
true_beta = 1
true_sigma = 1
n_trials = 200

ramp_model = RampModelHMM(K=K, beta=true_beta, sigma=true_sigma, dt=dt)
ramp_spike_trains = np.array([
    ramp_model.simulate_spikes(n_steps=T, initial_state=s0, R_h=fixed_rh, dt=dt)[2]
    for _ in range(n_trials)
])

step_model = StepModelHMM()
step_spike_trains = np.array([step_model.simulate_spikes(n_steps=T)[2]
                             for _ in range(n_trials)
])

In [ ]:
step_res = HMM_inference.step_inference_scan(
              spktrn_arg=ramp_spike_trains,
              R_low=10, R_high=50,
              M=45, plot=False)

ramp_res = HMM_inference.ramp_inference_scan(
              spktrn_arg=ramp_spike_trains,
              K=100, M=45, plot=False)

step_log_ml  = step_res[3]      # log p(D | step)
ramp_log_ml  = ramp_res[3]      # log p(D | ramp)


print("log BF =", ramp_log_ml - step_log_ml)
BF = np.exp(ramp_log_ml - step_log_ml)
print("BF     =", BF)


In [ ]:
T        = 500
dt       = 1 / T
n_trials = 200
R_low    = 10
R_high   = 50
M_grid   = 45                 # we gotta ensure same resolution for both scans


true_m = 250                  # smack in the middle of the scanner's grid
true_r = 4

step_model = StepModelHMM(m=true_m, r=true_r, dt=dt, exact=True)
step_spike_trains = np.array([
    step_model.simulate_spikes(n_steps=T, R_low=R_low, R_high=R_high, dt=dt)[2]
    for _ in range(n_trials)
])


step_res  = HMM_inference.step_inference_scan(
                spktrn_arg=step_spike_trains,
                R_low=R_low, R_high=R_high,
                M=M_grid, plot=False)

ramp_res  = HMM_inference.ramp_inference_scan(
                spktrn_arg=step_spike_trains,
                K=100, M=M_grid, plot=False)

log_e_step = step_res[3]      # log p(D | step)
log_e_ramp = ramp_res[3]      # log p(D | ramp)

log_BF = log_e_ramp - log_e_step
print("log BF =", log_BF)     


Profiling cell to check where the code is taking longest to run:

In [ ]:
import cProfile
import pstats
from io import StringIO

pr = cProfile.Profile()
pr.enable()

# Run the grid inference function
HMM_inference.ramp_inference_scan(
    true_beta=1,
    true_sigma=0.3,
    N=50,
    M=40,
)
pr.disable()

# Process stats
s = StringIO()
ps = pstats.Stats(pr, stream=s).strip_dirs().sort_stats("tottime")
ps.print_stats(20)  # Show top 20

# Display
print(s.getvalue())


# NEW: this generates the data we need to input into the scan functions for 3.2.

In [ ]:
import os
import pickle
import numpy as np
from spikedata import ramp_sample_prior_and_simulate_spikes as ramp_datagen
from spikedata import step_sample_prior_and_simulate_spikes as step_datagen

dataset_sizes = np.array([1, 5, 10, 100, 1e3, 1e4]).astype(int)
widths = np.array([0.25, 0.5, 0.75, 1])
fixed_N_for_vary_gaussian_width = 100

# Define dictionary names and initialize them to None
data_vars = {
    "uniform_ramp_data": None,
    "gaussian_ramp_data_vary_dataset_size": None,
    "gaussian_ramp_data_vary_width": None,
    "uniform_step_data": None,
    "gaussian_step_data_vary_dataset_size": None,
    "gaussian_step_data_vary_width": None,
}

# Try loading each dictionary from file if available
for name in data_vars:
    filename = f"{name}.pkl"
    if os.path.exists(filename):
        with open(filename, "rb") as f:
            data_vars[name] = pickle.load(f)

# Now generate any data that failed to load
if data_vars["uniform_ramp_data"] is None:
    data_vars["uniform_ramp_data"] = {}
    for N in dataset_sizes:
        data_vars["uniform_ramp_data"][N] = ramp_datagen(N=N, prior_type='uniform')

if data_vars["gaussian_ramp_data_vary_dataset_size"] is None:
    data_vars["gaussian_ramp_data_vary_dataset_size"] = {}
    for N in dataset_sizes:
        data_vars["gaussian_ramp_data_vary_dataset_size"][N] = ramp_datagen(N=N, prior_type='gaussian')

if data_vars["uniform_step_data"] is None:
    data_vars["uniform_step_data"] = {}
    for N in dataset_sizes:
        data_vars["uniform_step_data"][N] = step_datagen(N=N, prior_type='uniform')

if data_vars["gaussian_step_data_vary_dataset_size"] is None:
    data_vars["gaussian_step_data_vary_dataset_size"] = {}
    for N in dataset_sizes:
        data_vars["gaussian_step_data_vary_dataset_size"][N] = step_datagen(N=N, prior_type='gaussian')

if data_vars["gaussian_ramp_data_vary_width"] is None:
    data_vars["gaussian_ramp_data_vary_width"] = {}
    for sig in widths:
        data_vars["gaussian_ramp_data_vary_width"][sig] = ramp_datagen(
            N=fixed_N_for_vary_gaussian_width,
            prior_type='gaussian',
            prior_sd_fraction=sig
        )

if data_vars["gaussian_step_data_vary_width"] is None:
    data_vars["gaussian_step_data_vary_width"] = {}
    for sig in widths:
        data_vars["gaussian_step_data_vary_width"][sig] = step_datagen(
            N=fixed_N_for_vary_gaussian_width,
            prior_type='gaussian',
            prior_sd_fraction=sig
        )

# Now unpack from dict to globals if you want (optional)
uniform_ramp_data = data_vars["uniform_ramp_data"]
gaussian_ramp_data_vary_dataset_size = data_vars["gaussian_ramp_data_vary_dataset_size"]
gaussian_ramp_data_vary_width = data_vars["gaussian_ramp_data_vary_width"]
uniform_step_data = data_vars["uniform_step_data"]
gaussian_step_data_vary_dataset_size = data_vars["gaussian_step_data_vary_dataset_size"]
gaussian_step_data_vary_width = data_vars["gaussian_step_data_vary_width"]


In [ ]:
import pickle

# Your dictionaries
uniform_ramp_data = {}
gaussian_ramp_data_vary_dataset_size = {}
gaussian_ramp_data_vary_width = {}

uniform_step_data = {}
gaussian_step_data_vary_dataset_size = {}
gaussian_step_data_vary_width = {}

# Dictionary of variable names and their objects
dicts_to_save = {
    "uniform_ramp_data": uniform_ramp_data,
    "gaussian_ramp_data_vary_dataset_size": gaussian_ramp_data_vary_dataset_size,
    "gaussian_ramp_data_vary_width": gaussian_ramp_data_vary_width,
    "uniform_step_data": uniform_step_data,
    "gaussian_step_data_vary_dataset_size": gaussian_step_data_vary_dataset_size,
    "gaussian_step_data_vary_width": gaussian_step_data_vary_width,
}

# Save each one to its own pickle file
for name, obj in dicts_to_save.items():
    with open(f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f)


## Task 3.2.1 - error against dataset size

### Functions for sampling parameters and simulating dataset

In [ ]:
import functools, joblib
from scipy.special import logsumexp    

# ──────────────────────────────────────────────────────────────
# Draw true parameters from the *uniform* priors specified in §3
# ──────────────────────────────────────────────────────────────
def sample_ramp_params(T):
    beta  = np.random.uniform(0, 4)                       # β ∈ [0,4]
    lnσ   = np.random.uniform(np.log(0.04), np.log(4))    # log-σ  uniform
    sigma = np.exp(lnσ)
    return dict(beta=beta, sigma=sigma)

def sample_step_params(T):
    r = np.random.randint(1, 7)                 # r ∈ {1,…,6}
    m = np.random.uniform(T/4, 3*T/4)           # m ∈ [T/4, 3T/4]
    return dict(m=m, r=r)

# ──────────────────────────────────────────────────────────────
# Simulate a whole *dataset* (many trials) from one model
# ──────────────────────────────────────────────────────────────
def simulate_dataset(model, params, N_trials, T, R_low, R_high, R_h):
    dt = 1/T
    if model == 'ramp':
        hmm = RampModelHMM(K=100, dt=dt, **params)
        s0  = int(round(0.2*(hmm.K-1)))
        spikes = np.array([hmm.simulate_spikes(T, s0, R_h, dt)[2]
                           for _ in range(N_trials)])
    elif model == 'step':
        hmm = StepModelHMM(dt=dt, exact=True, **params)
        spikes = np.array([hmm.simulate_spikes(T, R_low, R_high, dt)[2]
                           for _ in range(N_trials)])
    return spikes


### Functions for building and classifying model

In [ ]:
FAST_M  = 10          # grid along each axis
FAST_K  = 8          # ramp discretisation
FAST_NJ = 1         # use all cores
# from task32 import build_ramp_grid, build_step_models

# ------------- one-time builders -------------
@functools.lru_cache(maxsize=None)
def build_ramp_grid(M, K, T, R_h):
    """Return (T_grid, lambdas) for the ramp model."""
    dt = 1 / T
    beta_vals   = np.linspace(0, 4, M)
    sigma_vals  = np.exp(np.linspace(np.log(0.04), np.log(4), M))
    x_grid      = np.arange(K) / (K - 1)

    T_grid = np.empty((M, M, K, K))
    for i, beta in enumerate(beta_vals):
        for j, sigma in enumerate(sigma_vals):
            T_grid[i, j] = RampModelHMM(K=K, beta=beta, sigma=sigma, dt=dt).T
    lambdas = R_h * x_grid * dt          # Poisson means used in ll
    return T_grid, lambdas

'''
@functools.lru_cache(maxsize=None)
def build_step_models(M, T, R_low, R_high):
    """Return the grid of step-model transition matrices and Poisson rates."""
    dt = 1 / T
    m_vals = np.linspace(0.25*T, 0.75*T, M)    # same rule the notebook uses
    r_vals = np.arange(1, 7)                   # {1,…,6}
    Ps_step = [[StepModelHMM(m=m, r=r, dt=dt, exact=True).T
                for r in r_vals] for m in m_vals]
    rates   = np.array([R_low*dt, R_high*dt])  # length-2
    return np.array(Ps_step), rates, m_vals, r_vals
'''

@functools.lru_cache(maxsize=None)
def build_step_models(M, T, R_low, R_high):
    """Return the grid of step-model transition matrices and Poisson rates."""
    dt = 1 / T
    m_vals = np.linspace(0.25*T, 0.75*T, M)    # same rule the notebook uses
    r_vals = np.arange(1, 7)                   # {1,…,6}
    # build a (len(m_vals), len(r_vals), 2, 2) tensor of 2-state transition matrices
    Ps_step = np.empty((len(m_vals), len(r_vals), 2, 2))
    for i, m in enumerate(m_vals):
        for j, r in enumerate(r_vals):
            Ps_step[i, j] = StepModelHMM(m=m, r=r, dt=dt, exact=False).T

    rates = np.array([R_low*dt, R_high*dt])      # λ_low, λ_high already in “spikes per bin”
    return Ps_step, rates, m_vals, r_vals




def classify_dataset_fast(spike_trains, T, R_low, R_high, R_h,
                          M=FAST_M, K=FAST_K,
                          prior='uniform', ramp_pre=None, step_pre=None):
    """Same API as before but ~10–30× faster."""

        # ··· obtain pre-computed grids (or reuse the ones handed in) ···
    if ramp_pre is None:
        ramp_pre = build_ramp_grid(M, K, T, R_h)
    if step_pre is None:
        step_pre = build_step_models(M, T, R_low, R_high)

    # ---------- Ramp -----------
    T_grid, lambdas = ramp_pre
    log_ml_ramp = HMM_inference.marginal_ll_ramp(spike_trains, T_grid, lambdas)

    # ---------- Step -----------
    Ps_step, rates_step, m_vals, r_vals = step_pre
    log_ml_step = HMM_inference.marginal_ll_step(
        spike_trains, Ps_step, rates_step, m_vals, r_vals)

    return 'ramp' if log_ml_ramp > log_ml_step else 'step'

### Function for computing classification error 

In [ ]:
import tqdm, collections, matplotlib.pyplot as plt

import joblib, numpy as np

def error_curve(N_trials_list,      # e.g. [25, 50, 100]
                n_datasets   = 50,  # replicates per point
                T            = 500,
                R_low        = 10,
                R_high       = 50,
                R_h          = 50,
                seed         = 1):

    rng = np.random.default_rng(seed)
    err_ramp, err_step = [], []

    # ------------------------------------------------------------
    # pre-compute the two grids once and reuse them everywhere
    ramp_pre = build_ramp_grid(FAST_M, FAST_K, T, R_h)
    step_pre = build_step_models(FAST_M, T, R_low, R_high)
    # ------------------------------------------------------------

    for N in N_trials_list:

        # --------- worker that handles ONE simulated dataset ---------
        def do_one_dataset(model_label):
            if model_label == 'ramp':
                p = sample_ramp_params(T)
            else:
                p = sample_step_params(T)

            spikes = simulate_dataset(model_label, p, N, T,
                                      R_low, R_high, R_h)

            return classify_dataset_fast(spikes, T, R_low, R_high, R_h,
                                         ramp_pre=ramp_pre,
                                         step_pre=step_pre)
        # ------------------------------------------------------------

        # run (n_datasets × 2 models) in parallel
        outs = joblib.Parallel(n_jobs=-1, backend='threading')(
                 joblib.delayed(do_one_dataset)(lab)
                 for lab in ['ramp', 'step'] * n_datasets)

        wrong_ramp = sum(o != 'ramp' for o in outs[0::2])
        wrong_step = sum(o != 'step' for o in outs[1::2])

        err_ramp.append(wrong_ramp / n_datasets)
        err_step.append(wrong_step / n_datasets)

    return np.asarray(err_ramp), np.asarray(err_step)


### Calling functions and generating plot

In [ ]:
N_list = [5, 10, 15, 20]
err_ramp, err_step = error_curve(N_list, n_datasets=50)

plt.figure(figsize=(6,4))
plt.plot(N_list, err_ramp, 'o-',  label='true ramp → mis-classified')
plt.plot(N_list, err_step, 's-',  label='true step → mis-classified')
plt.gca().invert_xaxis()  # optional: large-N at left
plt.xlabel('number of trials in dataset')
plt.ylabel('error rate')
plt.title('Bayesian model-selection error vs dataset size')
plt.legend()
plt.tight_layout()
plt.show()


Now, for 3.2.2, we can rerun the same code. Just tell the scans to use a gaussian prior, set the prior_sd_fraction, and ensure that the step model r prior mean is 1.